Install & Imports

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!mkdir -p /content/data/snli
!wget -O /content/data/snli/snli_1.0.zip \
https://nlp.stanford.edu/projects/snli/snli_1.0.zip


--2026-02-07 16:34:58--  https://nlp.stanford.edu/projects/snli/snli_1.0.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 94550081 (90M) [application/zip]
Saving to: ‘/content/data/snli/snli_1.0.zip’

/content/data/snli/ 100%[===================>]  90.17M  58.1MB/s    in 1.6s    

2026-02-07 16:35:00 (58.1 MB/s) - ‘/content/data/snli/snli_1.0.zip’ saved [94550081/94550081]



In [ ]:
!unzip -q /content/data/snli/snli_1.0.zip -d /content/data/snli


In [ ]:
!ls /content/data/snli/snli_1.0


Icon	    snli_1.0_dev.jsonl	snli_1.0_test.jsonl  snli_1.0_train.jsonl
README.txt  snli_1.0_dev.txt	snli_1.0_test.txt    snli_1.0_train.txt


In [ ]:
!mkdir -p /content/data/hatexplain
!git clone https://github.com/hate-alert/HateXplain.git /content/data/hatexplain


Cloning into '/content/data/hatexplain'...
remote: Enumerating objects: 414, done.
remote: Counting objects: 100% (188/188), done.
remote: Compressing objects: 100% (46/46), done.
remote: Total 414 (delta 159), reused 142 (delta 142), pack-reused 226 (from 1)
Receiving objects: 100% (414/414), 6.75 MiB | 10.54 MiB/s, done.
Resolving deltas: 100% (235/235), done.


In [ ]:
!ls /content


data  drive  sample_data


In [ ]:
!git clone https://github.com/hate-alert/HateXplain.git /content/hatexplain


Cloning into '/content/hatexplain'...
remote: Enumerating objects: 414, done.
remote: Counting objects: 100% (188/188), done.
remote: Compressing objects: 100% (46/46), done.
remote: Total 414 (delta 159), reused 142 (delta 142), pack-reused 226 (from 1)
Receiving objects: 100% (414/414), 6.75 MiB | 10.27 MiB/s, done.
Resolving deltas: 100% (235/235), done.


In [ ]:
!find /content/hatexplain -name "dataset.json"


/content/hatexplain/Data/dataset.json


In [ ]:
!ls /content/hatexplain


best_model_json			     Models
best_runs.sh			     Parameters_description.md
Bias_Calculation_NB.ipynb	     parameters_selection.py
convert_to_word2vec.py		     Preprocess
Data				     README.md
eraserbenchmark			     requirements.txt
Example_HateExplain.ipynb	     TensorDataset
Explainability_Calculation_NB.ipynb  testing_for_bias.py
Figures				     testing_with_lime.py
LICENSE				     testing_with_rational.py
manual_training_inference.py	     test_parallel.sh


In [ ]:
!pip install -q transformers datasets torch scikit-learn pandas tqdm nltk


In [ ]:
import torch
import numpy as np
import pandas as pd
import re
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW
from datasets import load_dataset
from tqdm import tqdm
import nltk

nltk.download("punkt")


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

Text cleaning

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()


In [ ]:
from torch.utils.data import Dataset
import torch

class NLPRationaleDataset(Dataset):
    def __init__(self, data, tokenizer, max_len=128):
        self.data = data
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        encoding = self.tokenizer(
            item["text"],
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label": torch.tensor(item["label"], dtype=torch.long)
        }


DATASETS

In [ ]:
import json

def load_snli_jsonl(path, limit=3000):
    texts, labels = [], []
    label_map = {"entailment": 0, "neutral": 1, "contradiction": 2}

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if len(texts) >= limit:
                break
            ex = json.loads(line)
            if ex["gold_label"] == "-":
                continue
            text = ex["sentence1"] + " [SEP] " + ex["sentence2"]
            texts.append(text)
            labels.append(label_map[ex["gold_label"]])

    return texts, labels


In [ ]:
import json

def load_hatexplain_local(json_path, limit=3000):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    texts, labels = [], []
    label_map = {"normal": 0, "offensive": 1, "hatespeech": 2}

    for post_id, ex in data.items():
        if len(texts) >= limit:
            break

        text = " ".join(ex["post_tokens"])
        # Fix: Iterate through the list of annotators to get their labels
        ann_labels = [annotator['label'] for annotator in ex["annotators"]]
        label = max(set(ann_labels), key=ann_labels.count)
        label = label_map[label]

        texts.append(text)
        labels.append(label)

    return texts, labels

In [ ]:
def load_sst2(limit=3000):
    dataset = load_dataset("glue", "sst2", split="train")

    texts = [clean_text(ex["sentence"]) for ex in dataset.select(range(limit))]
    labels = [int(ex["label"]) for ex in dataset.select(range(limit))]

    return texts, labels


Dataset class

In [ ]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.encodings = tokenizer(
            texts, padding=True, truncation=True, return_tensors="pt"
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item


Train BERT (baseline)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def train_bert(texts, labels, num_classes, epochs=2):
    tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
    dataset = TextDataset(texts, labels, tokenizer)
    dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

    model = BertForSequenceClassification.from_pretrained(
        "bert-base-uncased",
        num_labels=num_classes
    ).to(device)

    optimizer = AdamW(model.parameters(), lr=2e-5)

    model.train()
    for ep in range(epochs):
        total_loss = 0
        for batch in tqdm(dataloader):
            optimizer.zero_grad()
            outputs = model(
                input_ids=batch["input_ids"].to(device),
                attention_mask=batch["attention_mask"].to(device),
                labels=batch["labels"].to(device)
            )
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"Epoch {ep+1} | Avg Loss: {total_loss/len(dataloader):.4f}")

    return model, tokenizer


Gradient × Input (post‑hoc explanation)

In [ ]:
def gradient_input(model, input_ids, attention_mask, target_label):
    model.eval()

    embeddings = model.bert.embeddings(input_ids)
    embeddings.requires_grad_(True)

    outputs = model(
        inputs_embeds=embeddings,
        attention_mask=attention_mask
    )

    logit = outputs.logits[0, target_label]
    grads = torch.autograd.grad(logit, embeddings)[0]

    return grads


In [ ]:
def gradient_input(model, input_ids, attention_mask, labels):
    """
    Computes gradient-based token importance.
    Works for hx_model, sst_model, and snli_model.
    """

    model.zero_grad()
    model.train()  # ensures gradients are tracked

    # Get embeddings
    embeddings = model.bert.embeddings.word_embeddings(input_ids)
    embeddings.requires_grad_(True)
    embeddings.retain_grad()  # 🔥 CRITICAL FIX

    # Forward pass using embeddings
    outputs = model(
        inputs_embeds=embeddings,
        attention_mask=attention_mask,
        labels=labels
    )

    loss = outputs.loss
    loss.backward()

    grads = embeddings.grad  # now NOT None

    # Convert embedding gradients → token importance
    token_importance = torch.norm(grads, dim=-1)

    return token_importance


Faithfulness (deletion test)

In [ ]:
def deletion_faithfulness(model, tokenizer, text, label, k=3):
    tokens = tokenizer(text, return_tensors="pt")
    input_ids = tokens["input_ids"].to(device)
    attention_mask = tokens["attention_mask"].to(device)

    grads = gradient_input(model, input_ids, attention_mask, label)
    scores = grads.abs().sum(dim=-1).squeeze(0)

    top_k = torch.topk(scores, k=k).indices.tolist()
    masked_ids = input_ids.clone()
    for idx in top_k:
        masked_ids[0, idx] = tokenizer.pad_token_id

    with torch.no_grad():
        p_orig = torch.softmax(
            model(input_ids, attention_mask=attention_mask).logits, dim=1
        )
        p_mask = torch.softmax(
            model(masked_ids, attention_mask=attention_mask).logits, dim=1
        )

    return (p_orig[0, label] - p_mask[0, label]).item()


Run experiments

In [ ]:
snli_texts, snli_labels = load_snli_jsonl(
    "/content/data/snli/snli_1.0/snli_1.0_train.jsonl",
    limit=3000
)

snli_model, snli_tok = train_bert(snli_texts, snli_labels, num_classes=3)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
100%|██████████| 3000/3000 [03:43<00:

Epoch 1 | Avg Loss: 1.0204


100%|██████████| 3000/3000 [03:24<00:00, 14.67it/s]

Epoch 2 | Avg Loss: 0.7025


In [ ]:
hx_texts, hx_labels = load_hatexplain_local(
    "/content/hatexplain/Data/dataset.json",
    limit=3000
)

hx_model, hx_tok = train_bert(hx_texts, hx_labels, num_classes=3)



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
100%|██████████| 3000/3000 [03:26<00:

Epoch 1 | Avg Loss: 0.7453


100%|██████████| 3000/3000 [03:28<00:00, 14.40it/s]

Epoch 2 | Avg Loss: 0.5893


In [ ]:


# SST‑2
sst_texts, sst_labels = load_sst2(3000)
sst_model, sst_tok = train_bert(sst_texts, sst_labels, 2)


README.md: 0.00B [00:00, ?B/s]

sst2/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

sst2/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

sst2/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
100%|██████████| 3000/3000 [03:26<00:

Epoch 1 | Avg Loss: 0.4019


100%|██████████| 3000/3000 [03:27<00:00, 14.48it/s]

Epoch 2 | Avg Loss: 0.1809


Faithfulness Penalty Function (STABLE)

In [ ]:
def faithfulness_penalty(
    model,
    input_ids,
    attention_mask,
    labels,
    importance_scores,
    tokenizer,              # ✅ pass tokenizer explicitly
    top_k_ratio=0.2
):
    import torch.nn.functional as F

    batch_size, seq_len = input_ids.size()
    k = max(1, int(top_k_ratio * seq_len))

    # Original confidence
    with torch.no_grad():
        orig_logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        ).logits
        orig_probs = F.softmax(orig_logits, dim=-1)
        orig_conf = orig_probs[range(batch_size), labels]

    # Mask top-k important tokens
    masked_ids = input_ids.clone()
    masked_mask = attention_mask.clone()

    topk_idx = torch.argsort(
        importance_scores, dim=1, descending=True
    )[:, :k]

    for i in range(batch_size):
        masked_ids[i, topk_idx[i]] = tokenizer.pad_token_id  # ✅ FIXED
        masked_mask[i, topk_idx[i]] = 0

    # New confidence
    with torch.no_grad():
        new_logits = model(
            input_ids=masked_ids,
            attention_mask=masked_mask
        ).logits
        new_probs = F.softmax(new_logits, dim=-1)
        new_conf = new_probs[range(batch_size), labels]

    penalty = torch.mean(F.relu(new_conf - orig_conf))
    return penalty


In [ ]:
import torch.nn.functional as F

def faithfulness_penalty(
    model,
    tokenizer,              # ✅ ADD THIS
    input_ids,
    attention_mask,
    labels,
    importance_scores,
    top_k_ratio=0.2
):

    """
    Penalizes the model if removing important tokens
    does not reduce prediction confidence.
    """

    batch_size, seq_len = input_ids.size()
    k = max(1, int(top_k_ratio * seq_len))

    # Original confidence
    with torch.no_grad():
        orig_logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        ).logits
        orig_probs = F.softmax(orig_logits, dim=-1)
        orig_conf = orig_probs[range(batch_size), labels]

    # Mask top-k important tokens
    masked_ids = input_ids.clone()
    masked_mask = attention_mask.clone()

    topk_idx = torch.argsort(
        importance_scores, dim=1, descending=True
    )[:, :k]

    for i in range(batch_size):
        masked_ids[i, topk_idx[i]] = tokenizer.pad_token_id

        masked_mask[i, topk_idx[i]] = 0

    # New confidence
    new_logits = model(
        input_ids=masked_ids,
        attention_mask=masked_mask
    ).logits
    new_probs = F.softmax(new_logits, dim=-1)
    new_conf = new_probs[range(batch_size), labels]

    # Penalty: confidence should drop
    penalty = torch.mean(F.relu(new_conf - orig_conf))

    return penalty


Faithfulness-Aware Training Loop (FINAL)

In [ ]:
def train_one_epoch_faithful(
    model,
    dataloader,
    optimizer,
    lambda_faith=0.05
):
    model.train()
    total_loss = 0

    for batch in dataloader:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        # ----- TASK LOSS -----
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        task_loss = outputs.loss

        # ----- EXPLANATIONS (NO GRAD) -----
        with torch.no_grad():
            importance_scores = generate_token_importance(
                model, batch
            )

        # ----- FAITHFULNESS PENALTY -----
        penalty = faithfulness_penalty(
            model,
            input_ids,
            attention_mask,
            labels,
            importance_scores
        )

        # ----- TOTAL LOSS -----
        loss = task_loss + lambda_faith * penalty

        if torch.isnan(loss):
            continue

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)


Run Faithfulness-Aware Training (FINAL)

In [ ]:
def train_one_epoch_faithful(
    model,
    dataloader,
    optimizer,
    tokenizer,
    lambda_faith,
    top_k_ratio=0.2
):
    model.train()
    total_task_loss = 0.0
    total_penalty = 0.0

    for step, batch in enumerate(dataloader):
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        # ----- TASK LOSS -----
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        task_loss = outputs.loss

        # ----- FAITHFULNESS PENALTY (NO GRAD) -----
        with torch.no_grad():
            importance_scores = gradient_input(
                model,
                input_ids,
                attention_mask,
                labels
            )

            penalty = faithfulness_penalty(
                model=model,
                tokenizer=tokenizer,
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,
                importance_scores=importance_scores,
                top_k_ratio=top_k_ratio
            )

        # ----- BACKPROP -----
        task_loss.backward()
        optimizer.step()

        total_task_loss += task_loss.item()
        total_penalty += penalty.item()

        # 🔍 DEBUG PRINT (EVERY 10 BATCHES)
        if step % 10 == 0:
            print(
                f"[Batch {step}] "
                f"Task Loss: {task_loss.item():.4f} | "
                f"Faithfulness Penalty: {penalty.item():.4f}"
            )

    avg_task_loss = total_task_loss / len(dataloader)
    avg_penalty = total_penalty / len(dataloader)

    print("\n✅ Epoch Summary")
    print(f"Average Task Loss: {avg_task_loss:.4f}")
    print(f"Average Faithfulness Penalty: {avg_penalty:.4f}")

    return avg_task_loss


In [ ]:
# Freeze encoder ONLY
for param in hx_model.bert.parameters():
    param.requires_grad = False

# Unfreeze classifier head (CRITICAL)
for param in hx_model.classifier.parameters():
    param.requires_grad = True


In [ ]:
trainable = [p for p in hx_model.parameters() if p.requires_grad]
print("Trainable parameters:", len(trainable))


Trainable parameters: 2


In [ ]:
def train_one_epoch_faithful(model, dataloader, optimizer):
    model.train()
    total_loss = 0.0

    for batch in dataloader:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss  # ✅ requires_grad = True
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)


In [ ]:
from torch.optim import AdamW
from torch.utils.data import DataLoader

hx_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
hx_dataset = TextDataset(hx_texts, hx_labels, hx_tokenizer)
hx_dataloader = DataLoader(hx_dataset, batch_size=1, shuffle=True)

# Ensure the model's classifier head is unfrozen and create optimizer
for param in hx_model.bert.parameters():
    param.requires_grad = False
for param in hx_model.classifier.parameters():
    param.requires_grad = True
faith_optimizer = AdamW(
    filter(lambda p: p.requires_grad, hx_model.parameters()),
    lr=2e-5
)

epochs = 2

for epoch in range(epochs):
    loss = train_one_epoch_faithful(
        model=hx_model,
        dataloader=hx_dataloader,
        optimizer=faith_optimizer
    )
    print(f"[Epoch {epoch+1}] Loss: {loss:.4f}")

[Epoch 1] Loss: 0.4165
[Epoch 2] Loss: 0.4142


In [ ]:
hx_model.eval()  # evaluation mode is OK

batch = next(iter(hx_dataloader))

input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)
labels = batch["labels"].to(device)

# IMPORTANT: enable gradients for explanation
importance_scores = gradient_input(
    hx_model,
    input_ids,
    attention_mask,
    labels
)

print("Importance shape:", importance_scores.shape)


Importance shape: torch.Size([1, 105])


In [ ]:
def gradient_input(model, input_ids, attention_mask, labels):
    model.zero_grad()

    embeddings = model.bert.embeddings.word_embeddings(input_ids)
    embeddings.requires_grad_(True)

    outputs = model(
        inputs_embeds=embeddings,
        attention_mask=attention_mask,
        labels=labels
    )

    loss = outputs.loss
    loss.backward()

    grads = embeddings.grad                     # [B, T, 768]
    token_importance = torch.norm(grads, dim=-1)  # [B, T]

    return token_importance


In [ ]:
def gradient_input(model, input_ids, attention_mask, labels):
    model.eval()
    model.zero_grad()

    # 1️⃣ Get embeddings
    embeddings = model.bert.embeddings(input_ids)
    embeddings.retain_grad()   # ✅ CRITICAL FIX

    # 2️⃣ Forward pass using embeddings
    outputs = model(
        inputs_embeds=embeddings,
        attention_mask=attention_mask,
        labels=labels
    )

    loss = outputs.loss

    # 3️⃣ Backward pass
    loss.backward()

    # 4️⃣ Get gradients w.r.t embeddings
    grads = embeddings.grad    # ✅ NOW THIS EXISTS

    # 5️⃣ Gradient × Input
    token_importance = torch.norm(grads * embeddings, dim=-1)

    return token_importance


In [ ]:
def gradient_input(model, input_ids, attention_mask, labels):
    model.eval()
    model.zero_grad()

    # 1️⃣ Get embeddings
    embeddings = model.bert.embeddings(input_ids)

    # ✅ CRITICAL FIXES
    embeddings.requires_grad_(True)
    embeddings.retain_grad()

    # 2️⃣ Forward pass using embeddings
    outputs = model(
        inputs_embeds=embeddings,
        attention_mask=attention_mask,
        labels=labels
    )

    loss = outputs.loss

    # 3️⃣ Backward pass
    loss.backward()

    # 4️⃣ Gradient × Input
    grads = embeddings.grad                     # [B, T, 768]
    token_importance = torch.norm(grads * embeddings, dim=-1)

    return token_importance


In [ ]:
hx_model.eval()  # evaluation mode is OK

batch = next(iter(hx_dataloader))

input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)
labels = batch["labels"].to(device)

# IMPORTANT: enable gradients for explanation
importance_scores = gradient_input(
    hx_model,
    input_ids,
    attention_mask,
    labels
)

print("Importance shape:", importance_scores.shape)


Importance shape: torch.Size([1, 105])


RESULTS

In [ ]:
import torch
import torch.nn.functional as F

def run_faithfulness_results_demo(
    model,
    tokenizer,
    dataset,
    sample_idx=0,
    top_k_ratio=0.2,
    dataset_name="HateXplain"
):
    model.eval()

    # --------------------------------------------------
    # 1. GET A REAL TRAINING SAMPLE (TOKENIZED)
    # --------------------------------------------------
    sample = dataset[sample_idx]
    input_ids = sample["input_ids"].unsqueeze(0).to(device)
    attention_mask = sample["attention_mask"].unsqueeze(0).to(device)
    labels = sample["labels"].unsqueeze(0).to(device)

    # Decode text for display
    decoded_text = tokenizer.decode(
        input_ids[0],
        skip_special_tokens=True
    )

    # --------------------------------------------------
    # 2. ORIGINAL PREDICTION
    # --------------------------------------------------
    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        probs = F.softmax(outputs.logits, dim=-1)
        original_conf = probs[0, labels.item()].item()
        original_pred = torch.argmax(probs, dim=-1).item()

    # --------------------------------------------------
    # 3. TOKEN IMPORTANCE (GRADIENT × INPUT)
    # --------------------------------------------------
    importance_scores = gradient_input(
        model,
        input_ids,
        attention_mask,
        labels
    )  # shape: [1, seq_len]

    scores = importance_scores[0]
    seq_len = scores.size(0)
    k = max(1, int(top_k_ratio * seq_len))

    topk_idx = torch.topk(scores, k=k).indices

    # --------------------------------------------------
    # 4. MASK IMPORTANT TOKENS (DELETION TEST)
    # --------------------------------------------------
    masked_ids = input_ids.clone()
    masked_mask = attention_mask.clone()

    masked_ids[0, topk_idx] = tokenizer.pad_token_id
    masked_mask[0, topk_idx] = 0

    # --------------------------------------------------
    # 5. PREDICTION AFTER MASKING
    # --------------------------------------------------
    with torch.no_grad():
        masked_outputs = model(
            input_ids=masked_ids,
            attention_mask=masked_mask
        )
        masked_probs = F.softmax(masked_outputs.logits, dim=-1)
        masked_conf = masked_probs[0, labels.item()].item()

    # --------------------------------------------------
    # 6. FAITHFULNESS PENALTY (NOVELTY)
    # --------------------------------------------------
    confidence_drop = original_conf - masked_conf
    penalty = max(0.0, -confidence_drop)

    # --------------------------------------------------
    # 7. PRINT RESULTS (FOR PPT / REPORT)
    # --------------------------------------------------
    print(f"\n================ {dataset_name} =================")
    print("Sentence:")
    print(decoded_text)
    print("-" * 60)
    print(f"Original prediction: {original_pred}")
    print(f"Original confidence: {original_conf:.4f}")
    print("-" * 60)
    print("Top important tokens:")
    for idx in topk_idx[:10]:
        token = tokenizer.decode(input_ids[0, idx])
        print(f"{token:<15} {scores[idx].item():.4f}")
    print("-" * 60)
    print("After deleting important tokens:")
    print(tokenizer.decode(masked_ids[0], skip_special_tokens=True))
    print(f"Confidence after deletion: {masked_conf:.4f}")
    print(f"Confidence drop: {confidence_drop:.4f}")
    print("-" * 60)
    print(f"Faithfulness penalty (NOVELTY): {penalty:.4f}")
    print("=" * 60)


In [ ]:
run_faithfulness_results_demo(
    model=hx_model,
    tokenizer=hx_tokenizer,
    dataset=hx_dataset,
    sample_idx=0,
    dataset_name="HateXplain"
)



================ HateXplain =================
Sentence:
i dont think im getting my baby them white 9 he has two white j and nikes not even touched
------------------------------------------------------------
Original prediction: 0
Original confidence: 0.9772
------------------------------------------------------------
Top important tokens:
nike            0.0448
baby            0.0201
9               0.0172
##s             0.0158
im              0.0155
j               0.0152
white           0.0138
white           0.0137
he              0.0110
and             0.0110
------------------------------------------------------------
After deleting important tokens:
i don even
Confidence after deletion: 0.9847
Confidence drop: -0.0075
------------------------------------------------------------
Faithfulness penalty (NOVELTY): 0.0075


In [ ]:
sorted([k for k in globals().keys() if not k.startswith("_")])


['AdamW',
 'BertForSequenceClassification',
 'BertTokenizer',
 'DataLoader',
 'Dataset',
 'F',
 'In',
 'NLPRationaleDataset',
 'Out',
 'TextDataset',
 'attention_mask',
 'batch',
 'clean_text',
 'deletion_faithfulness',
 'device',
 'drive',
 'epoch',
 'epochs',
 'exit',
 'faith_optimizer',
 'faithfulness_penalty',
 'get_ipython',
 'gradient_input',
 'hx_dataloader',
 'hx_dataset',
 'hx_labels',
 'hx_model',
 'hx_texts',
 'hx_tok',
 'hx_tokenizer',
 'importance_scores',
 'input_ids',
 'json',
 'labels',
 'load_dataset',
 'load_hatexplain_local',
 'load_snli_jsonl',
 'load_sst2',
 'loss',
 'nltk',
 'np',
 'param',
 'pd',
 'quit',
 're',
 'run_faithfulness_demo',
 'run_faithfulness_results_demo',
 'run_faithfulness_results_demo_esnli',
 'run_faithfulness_results_demo_sst2',
 'snli_enc',
 'snli_labels',
 'snli_model',
 'snli_sample',
 'snli_text',
 'snli_texts',
 'snli_tok',
 'sst_labels',
 'sst_model',
 'sst_sample',
 'sst_texts',
 'sst_tok',
 'torch',
 'tqdm',
 'train_bert',
 'train_one_

In [ ]:
# ================= HATEXPLAIN RESULTS =================

sample = hx_dataset[0]  # already tokenized

input_ids = sample["input_ids"].unsqueeze(0).to(device)
attention_mask = sample["attention_mask"].unsqueeze(0).to(device)
labels = sample["labels"].unsqueeze(0).to(device)

decoded_text = hx_tokenizer.decode(
    input_ids[0], skip_special_tokens=True
)

# Original prediction
with torch.no_grad():
    outputs = hx_model(input_ids=input_ids, attention_mask=attention_mask)
    probs = F.softmax(outputs.logits, dim=-1)
    orig_conf = probs[0, labels.item()].item()
    orig_pred = torch.argmax(probs, dim=-1).item()

# Explanation
importance_scores = gradient_input(
    hx_model, input_ids, attention_mask, labels
)

# Faithfulness penalty
penalty = faithfulness_penalty(
    model=hx_model,
    tokenizer=hx_tokenizer,
    input_ids=input_ids,
    attention_mask=attention_mask,
    labels=labels,
    importance_scores=importance_scores
)

print("\n================ HateXplain ================")
print("Sentence:")
print(decoded_text)
print(f"Original prediction: {orig_pred}")
print(f"Original confidence: {orig_conf:.4f}")
print(f"Faithfulness penalty (NOVELTY): {penalty:.4f}")
print("============================================")



================ HateXplain ================
Sentence:
i dont think im getting my baby them white 9 he has two white j and nikes not even touched
Original prediction: 0
Original confidence: 0.9772
Faithfulness penalty (NOVELTY): 0.0075


In [ ]:
# ================= E-SNLI RESULTS =================

idx = 0
text = snli_texts[idx]
label = torch.tensor([snli_labels[idx]]).to(device)

encoded = snli_tok(
    text,
    return_tensors="pt",
    padding="max_length",
    truncation=True,
    max_length=128
)

input_ids = encoded["input_ids"].to(device)
attention_mask = encoded["attention_mask"].to(device)

# Original prediction
with torch.no_grad():
    outputs = snli_model(input_ids=input_ids, attention_mask=attention_mask)
    probs = F.softmax(outputs.logits, dim=-1)
    orig_conf = probs[0, label.item()].item()
    orig_pred = torch.argmax(probs, dim=-1).item()

# Explanation
importance_scores = gradient_input(
    snli_model, input_ids, attention_mask, label
)

# Faithfulness penalty
penalty = faithfulness_penalty(
    model=snli_model,
    tokenizer=snli_tok,
    input_ids=input_ids,
    attention_mask=attention_mask,
    labels=label,
    importance_scores=importance_scores
)

print("\n================ E-SNLI ====================")
print("Sentence (Premise [SEP] Hypothesis):")
print(text)
print(f"Original prediction: {orig_pred}")
print(f"Original confidence: {orig_conf:.4f}")
print(f"Faithfulness penalty (NOVELTY): {penalty:.4f}")
print("============================================")



================ E-SNLI ====================
Sentence (Premise [SEP] Hypothesis):
A person on a horse jumps over a broken down airplane. [SEP] A person is training his horse for a competition.
Original prediction: 1
Original confidence: 0.8952
Faithfulness penalty (NOVELTY): 0.0000


In [ ]:
# ================= SST-2 RESULTS =================

idx = 0
text = sst_texts[idx]
label = torch.tensor([sst_labels[idx]]).to(device)

encoded = sst_tok(
    text,
    return_tensors="pt",
    padding="max_length",
    truncation=True,
    max_length=128
)

input_ids = encoded["input_ids"].to(device)
attention_mask = encoded["attention_mask"].to(device)

# Original prediction
with torch.no_grad():
    outputs = sst_model(input_ids=input_ids, attention_mask=attention_mask)
    probs = F.softmax(outputs.logits, dim=-1)
    orig_conf = probs[0, label.item()].item()
    orig_pred = torch.argmax(probs, dim=-1).item()

# Explanation
importance_scores = gradient_input(
    sst_model, input_ids, attention_mask, label
)

# Faithfulness penalty
penalty = faithfulness_penalty(
    model=sst_model,
    tokenizer=sst_tok,
    input_ids=input_ids,
    attention_mask=attention_mask,
    labels=label,
    importance_scores=importance_scores
)

print("\n================ SST-2 =====================")
print("Sentence:")
print(text)
print(f"Original prediction: {orig_pred}")
print(f"Original confidence: {orig_conf:.4f}")
print(f"Faithfulness penalty (NOVELTY): {penalty:.4f}")
print("============================================")




================ SST-2 =====================
Sentence:
hide new secretions from the parental units
Original prediction: 0
Original confidence: 0.9892
Faithfulness penalty (NOVELTY): 0.0000


In [ ]:
import torch
import torch.nn.functional as F
from torch.optim import AdamW

# -------------------------------
# STEP 0: SETUP
# -------------------------------
model = hx_model
tokenizer = hx_tokenizer
model.train()

sentence = "You are a disgusting and horrible person"
true_label = torch.tensor([1]).to(device)   # assume "hate/offensive" class

encoded = tokenizer(
    sentence,
    return_tensors="pt",
    padding="max_length",
    truncation=True,
    max_length=128
)

input_ids = encoded["input_ids"].to(device)
attention_mask = encoded["attention_mask"].to(device)

optimizer = AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=2e-5
)

# -------------------------------
# STEP 1: FORWARD PASS (BEFORE)
# -------------------------------
with torch.no_grad():
    outputs_before = model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )
    logits_before = outputs_before.logits
    probs_before = F.softmax(logits_before, dim=-1)
    conf_before = probs_before[0, true_label.item()].item()

print("\nSentence:")
print(sentence)
print("-" * 60)
print("Logits BEFORE update:")
print(logits_before)
print(f"True-label confidence BEFORE update: {conf_before:.6f}")

# -------------------------------
# STEP 2: EXPLANATION (Gradient × Input)
# -------------------------------
importance_scores = gradient_input(
    model,
    input_ids,
    attention_mask,
    true_label
)

scores = importance_scores[0]
topk = torch.topk(scores, k=5).indices

print("-" * 60)
print("Top important tokens:")
for idx in topk:
    token = tokenizer.decode(input_ids[0, idx])
    print(f"{token:<15} {scores[idx].item():.4f}")

# -------------------------------
# STEP 3: DELETION / MASKING TEST
# -------------------------------
masked_ids = input_ids.clone()
masked_mask = attention_mask.clone()

masked_ids[0, topk] = tokenizer.pad_token_id
masked_mask[0, topk] = 0

with torch.no_grad():
    outputs_masked = model(
        input_ids=masked_ids,
        attention_mask=masked_mask
    )
    probs_masked = F.softmax(outputs_masked.logits, dim=-1)
    conf_after_mask = probs_masked[0, true_label.item()].item()

confidence_drop = conf_before - conf_after_mask

print("-" * 60)
print("Sentence AFTER deleting important tokens:")
print(tokenizer.decode(masked_ids[0], skip_special_tokens=True))
print(f"Confidence AFTER deletion: {conf_after_mask:.6f}")
print(f"Confidence DROP (faithfulness signal): {confidence_drop:.6f}")

# -------------------------------
# STEP 4: BACKPROPAGATION STEP
# -------------------------------
optimizer.zero_grad()

outputs_train = model(
    input_ids=input_ids,
    attention_mask=attention_mask,
    labels=true_label
)

loss = outputs_train.loss
print("-" * 60)
print(f"Loss used for backprop: {loss.item():.6f}")

loss.backward()
optimizer.step()

# -------------------------------
# STEP 5: FORWARD PASS (AFTER UPDATE)
# -------------------------------
with torch.no_grad():
    outputs_after = model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )
    logits_after = outputs_after.logits
    probs_after = F.softmax(logits_after, dim=-1)
    conf_after = probs_after[0, true_label.item()].item()

print("-" * 60)
print("Logits AFTER update:")
print(logits_after)
print(f"True-label confidence AFTER update: {conf_after:.6f}")
print(f"Confidence CHANGE due to training: {conf_after - conf_before:.6f}")

print("-" * 60)
print("Logit difference (AFTER − BEFORE):")
print(logits_after - logits_before)
print("=" * 70)



Sentence:
You are a disgusting and horrible person
------------------------------------------------------------
Logits BEFORE update:
tensor([[ 2.9097, -0.7932, -1.6609]], device='cuda:0')
True-label confidence BEFORE update: 0.023817
------------------------------------------------------------
Top important tokens:
disgusting      0.2090
you             0.1734
[SEP]           0.1697
horrible        0.1556
and             0.1242
------------------------------------------------------------
Sentence AFTER deleting important tokens:
are a person
Confidence AFTER deletion: 0.020641
Confidence DROP (faithfulness signal): 0.003176
------------------------------------------------------------
Loss used for backprop: 3.697481
------------------------------------------------------------
Logits AFTER update:
tensor([[ 3.0315, -0.6149, -2.0846]], device='cuda:0')
True-label confidence AFTER update: 0.025274
Confidence CHANGE due to training: 0.001457
----------------------------------------------

In [ ]:
import torch
import torch.nn.functional as F
from torch.optim import AdamW

# --------------------------------------------------
# SETUP
# --------------------------------------------------
model = sst_model
tokenizer = sst_tok
model.train()

sentence = "The movie was not bad at all"
true_label = torch.tensor([1]).to(device)   # positive sentiment

encoded = tokenizer(
    sentence,
    return_tensors="pt",
    padding="max_length",
    truncation=True,
    max_length=128
)

input_ids = encoded["input_ids"].to(device)
attention_mask = encoded["attention_mask"].to(device)

optimizer = AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=2e-5
)

# --------------------------------------------------
# STEP 1: PREDICTION BEFORE TRAINING
# --------------------------------------------------
with torch.no_grad():
    outputs_before = model(input_ids=input_ids, attention_mask=attention_mask)
    logits_before = outputs_before.logits
    probs_before = F.softmax(logits_before, dim=-1)
    conf_before = probs_before[0, true_label.item()].item()

print("\nSentence:")
print(sentence)
print("-" * 60)
print("Logits BEFORE update:")
print(logits_before)
print(f"True-label confidence BEFORE update: {conf_before:.6f}")

# --------------------------------------------------
# STEP 2: EXPLANATION (LIKELY UNFAITHFUL)
# --------------------------------------------------
importance_scores = gradient_input(
    model,
    input_ids,
    attention_mask,
    true_label
)

scores = importance_scores[0]
topk = torch.topk(scores, k=5).indices

print("-" * 60)
print("Top important tokens (EXPLANATION):")
for idx in topk:
    token = tokenizer.decode(input_ids[0, idx])
    print(f"{token:<15} {scores[idx].item():.4f}")

# --------------------------------------------------
# STEP 3: DELETION TEST (UNFAITHFULNESS)
# --------------------------------------------------
masked_ids = input_ids.clone()
masked_mask = attention_mask.clone()

masked_ids[0, topk] = tokenizer.pad_token_id
masked_mask[0, topk] = 0

with torch.no_grad():
    masked_outputs = model(
        input_ids=masked_ids,
        attention_mask=masked_mask
    )
    masked_probs = F.softmax(masked_outputs.logits, dim=-1)
    conf_after_mask = masked_probs[0, true_label.item()].item()

confidence_drop = conf_before - conf_after_mask

print("-" * 60)
print("Sentence AFTER deleting important tokens:")
print(tokenizer.decode(masked_ids[0], skip_special_tokens=True))
print(f"Confidence AFTER deletion: {conf_after_mask:.6f}")
print(f"Confidence DROP: {confidence_drop:.6f}")

if abs(confidence_drop) < 0.001:
    print("⚠️ Explanation is UNFAITHFUL (prediction unchanged)")

# --------------------------------------------------
# STEP 4: BACKPROPAGATION (LEARNING)
# --------------------------------------------------
optimizer.zero_grad()

outputs_train = model(
    input_ids=input_ids,
    attention_mask=attention_mask,
    labels=true_label
)

loss = outputs_train.loss
print("-" * 60)
print(f"Loss used for backprop: {loss.item():.6f}")

loss.backward()
optimizer.step()

# --------------------------------------------------
# STEP 5: PREDICTION AFTER TRAINING
# --------------------------------------------------
with torch.no_grad():
    outputs_after = model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )
    logits_after = outputs_after.logits
    probs_after = F.softmax(logits_after, dim=-1)
    conf_after = probs_after[0, true_label.item()].item()

print("-" * 60)
print("Logits AFTER update:")
print(logits_after)
print(f"True-label confidence AFTER update: {conf_after:.6f}")
print(f"Confidence CHANGE due to training: {conf_after - conf_before:.6f}")

print("-" * 60)
print("Logit difference (AFTER − BEFORE):")
print(logits_after - logits_before)
print("=" * 70)



Sentence:
The movie was not bad at all
------------------------------------------------------------
Logits BEFORE update:
tensor([[ 1.9635, -1.2779]], device='cuda:0')
True-label confidence BEFORE update: 0.037637
------------------------------------------------------------
Top important tokens (EXPLANATION):
bad             1.1644
not             1.0953
was             0.5236
at              0.4309
the             0.3164
------------------------------------------------------------
Sentence AFTER deleting important tokens:
movie all
Confidence AFTER deletion: 0.399458
Confidence DROP: -0.361821
------------------------------------------------------------
Loss used for backprop: 2.747735
------------------------------------------------------------
Logits AFTER update:
tensor([[-1.5177,  1.7592]], device='cuda:0')
True-label confidence AFTER update: 0.963630
Confidence CHANGE due to training: 0.925993
------------------------------------------------------------
Logit difference (AFTER −